In [ ]:
import pandas as pd
from datasets import load_dataset

print("1. Downloading dataset...")
dataset = load_dataset("tner/bc5cdr")
df = pd.DataFrame(dataset['train'])

nodes = []
edges = []

tag_map = {1: "Chemical", 2: "Disease", 3: "Disease", 4: "Chemical"}

print("2. Extracting nodes, chunks, and edges...")
for idx, row in df.iterrows():
    tokens = row['tokens']
    tags = row['tags']
    
    text_chunk = " ".join(tokens)
    chunk_id = f"chunk_{idx}"
    
    nodes.append({"id": chunk_id, "label": "TextChunk", "name": text_chunk})
    
    current_entity = []
    current_type = None
    entities_in_sentence = []

    for token, tag in zip(tokens, tags):
        if tag in tag_map:
            current_entity.append(token)
            current_type = tag_map[tag]
        elif current_entity:
            entity_name = " ".join(current_entity)
            entities_in_sentence.append({"name": entity_name, "type": current_type})
            nodes.append({"id": entity_name, "label": current_type, "name": entity_name})
            current_entity = []
            current_type = None
            
    if current_entity:
        entity_name = " ".join(current_entity)
        entities_in_sentence.append({"name": entity_name, "type": current_type})
        nodes.append({"id": entity_name, "label": current_type, "name": entity_name})
            
    unique_entities = list(set([e['name'] for e in entities_in_sentence]))
    chemicals = [e['name'] for e in entities_in_sentence if e['type'] == 'Chemical']
    diseases = [e['name'] for e in entities_in_sentence if e['type'] == 'Disease']
    
    for entity in unique_entities:
        edges.append({
            "source": entity, 
            "relation": "MENTIONED_IN", 
            "target": chunk_id
        })
    
    for chem in set(chemicals):
        for dis in set(diseases):
            edges.append({
                "source": chem, 
                "relation": "CO_OCCURS_WITH", 
                "target": dis
            })

print("3. Deduplicating and saving to CSV...")
df_nodes = pd.DataFrame(nodes).drop_duplicates(subset=['id'])
df_edges = pd.DataFrame(edges).drop_duplicates()

df_nodes.to_csv("nodes_with_text.csv", index=False)
df_edges.to_csv("edges_with_text.csv", index=False)

print(f"Success! Saved {len(df_nodes)} unique Nodes (including Text Chunks).")
print(f"Success! Saved {len(df_edges)} unique Edges.")

1. Downloading dataset...
2. Extracting nodes, chunks, and edges...
3. Deduplicating and saving to CSV...
Success! Saved 8002 unique Nodes (including Text Chunks).
Success! Saved 11785 unique Edges.
